## MODEL A ( All Features )

### 1. Import the libraries

In [8]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

import joblib


### 2. Load the dataset

In [9]:
data = pd.read_csv(r"C:\Users\HP\Documents\PycharmProjects\network-anomaly-detector\Detector\data\processed-data\clean-data.csv", low_memory=False)
print(data.shape)
data.sample(10)

(2540047, 45)


,sport,dsport,proto,state,dur,sbytes,dbytes,sttl,dttl,service,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
146147,56523.0,22.0,tcp,FIN,6.516092,25936,49682,31,29,unknown,...,0,3,11,8,1,1,1,1,NaN,0
2497701,61027.0,7108.0,udp,CON,0.001734,528,304,31,29,unknown,...,0,5,4,2,3,1,1,2,NaN,0
528179,1730.0,18143.0,tcp,FIN,0.015887,2854,29648,31,29,unknown,...,0,2,5,3,2,1,1,1,NaN,0
2370602,45828.0,6881.0,tcp,FIN,10.407466,36762,1641360,31,29,unknown,...,0,13,12,3,3,2,1,7,NaN,0
1212789,40718.0,53.0,udp,CON,0.001036,146,178,31,29,dns,...,0,1,3,1,2,1,1,1,NaN,0
1712124,19440.0,143.0,tcp,FIN,0.036835,7818,15588,31,29,unknown,...,0,2,4,2,2,1,1,1,NaN,0
1424521,1043.0,53.0,udp,INT,0.000006,264,0,60,0,dns,...,0,30,30,9,9,9,9,30,NaN,0
673956,39199.0,6333.0,tcp,FIN,0.011888,2542,21294,31,29,unknown,...,0,10,9,6,4,1,1,2,NaN,0
1270889,13281.0,53.0,udp,CON,0.001008,146,178,31,29,dns,...,0,3,1,4,3,2,1,2,NaN,0
1413829,1043.0,53.0,udp,INT,0.000007,264,0,60,0,dns,...,0,36,36,36,36,36,16,36,NaN,0


### 3. Train-Test-Split

In [10]:
X_train,X_test,y_train,y_test = train_test_split(
    data.drop(columns=["Label","attack_cat"]),
    data["Label"],
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(2032037, 43)
(508010, 43)
(2032037,)
(508010,)


attack_cat was dropped because it will be used in the next model, not needed here

### 4. Separate the diffent type of features

In [11]:
numeric_cols = X_train.select_dtypes(
    include=[np.number]
).columns
categorical_cols = X_train.select_dtypes(
    exclude=[np.number]
).columns

print("Numeric Features :\n",numeric_cols)
print("\nCategorical Features:\n",categorical_cols)

Numeric Features :
 Index(['sport', 'dsport', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'Sload',
       'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz',
       'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime',
       'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat',
       'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
       'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm'],
      dtype='str')

Categorical Features:
 Index(['proto', 'state', 'service'], dtype='str')


Making custom transformer for grouping low frequency categories

In [33]:
from sklearn.base import BaseEstimator, TransformerMixin

class RareCategories(BaseEstimator, TransformerMixin):
    def __init__(self, top_n=7):
        self.top_n = top_n
        self.top_categories_ = {}

    def fit(self, X, y=None):
        for col in X.columns:
            self.top_categories_[col] = (
                X[col].value_counts()
                .head(self.top_n)
                .index
                .tolist()
            )
        return self

    def transform(self, X):
        X = X.copy()

        for col in X.columns:
            X[col] = X[col].where(
                X[col].isin(self.top_categories_[col]),
                "other"
            )

        return X
    def get_feature_names_out(self, input_features=None):
        return input_features

### 5. Creating pipelines for Feature Engineering

In [34]:
num_pipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy='most_frequent')),
    ('Scaler', StandardScaler()),
])
cat_pipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy='most_frequent').set_output(transform = 'pandas')),
    ('Grouping', RareCategories(top_n=7)),
    ('Encoder', OneHotEncoder(handle_unknown='ignore'))
])

### 6. Using Column Transformer

In [35]:
preprocessor = ColumnTransformer([
    ("num", num_pipeline, numeric_cols),
    ("cat", cat_pipeline, categorical_cols),
])

#### *** Final Dataset ***

In [37]:
X_train_t = preprocessor.fit_transform(X_train)

print(X_train_t.shape)

X_train_prev = pd.DataFrame(
    X_train_t[:5],
    columns=preprocessor.get_feature_names_out()
)

pd.set_option('display.max_columns', None)
X_train_prev

(2032037, 64)


,num__sport,num__dsport,num__dur,num__sbytes,num__dbytes,num__sttl,num__dttl,num__Sload,num__Dload,num__Spkts,num__Dpkts,num__swin,num__dwin,num__stcpb,num__dtcpb,num__smeansz,num__dmeansz,num__trans_depth,num__res_bdy_len,num__Sjit,num__Djit,num__Stime,num__Ltime,num__Sintpkt,num__Dintpkt,num__tcprtt,num__synack,num__ackdat,num__is_sm_ips_ports,num__ct_state_ttl,num__ct_flw_http_mthd,num__is_ftp_login,num__ct_ftp_cmd,num__ct_srv_src,num__ct_srv_dst,num__ct_dst_ltm,num__ct_src_ ltm,num__ct_src_dport_ltm,num__ct_dst_sport_ltm,num__ct_dst_src_ltm,cat__proto_arp,cat__proto_icmp,cat__proto_ospf,cat__proto_other,cat__proto_sctp,cat__proto_tcp,cat__proto_udp,cat__proto_unas,cat__state_CLO,cat__state_CON,cat__state_ECO,cat__state_FIN,cat__state_INT,cat__state_REQ,cat__state_RST,cat__state_other,cat__service_dns,cat__service_ftp,cat__service_ftp-data,cat__service_http,cat__service_other,cat__service_smtp,cat__service_ssh,cat__service_unknown
0,1.190107,2.630674,-0.041828,-0.032874,-0.080924,-0.425967,-0.041412,-0.302206,1.811054,0.089112,-0.006173,0.835803,0.838098,-0.311972,-0.311667,-0.396788,0.838059,-0.2411,-0.0892,-0.093593,-0.212929,-1.155095,-1.155096,-0.069265,-0.054619,-0.118118,-0.104612,-0.114270,-0.040545,-0.382269,-0.198759,-0.130167,-0.111626,-0.203833,-0.461206,-0.421514,-0.109933,-0.429809,-0.420068,-0.519454,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.360893,1.786450,-0.041371,-0.013062,0.058359,-0.425967,-0.041412,-0.301914,2.813453,0.354600,0.158594,0.835803,0.838098,-0.665791,0.863902,-0.423120,1.380196,-0.2411,-0.0892,-0.092253,-0.213859,-1.155238,-1.155239,-0.069278,-0.054644,-0.105860,-0.105927,-0.089014,-0.040545,-0.382269,-0.198759,-0.130167,-0.111626,-0.203833,0.093264,-0.544015,-0.475491,-0.429809,-0.420068,-0.519454,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.873440,-0.380920,-0.041895,-0.029030,-0.054908,-0.425967,-0.041412,-0.300820,2.416547,0.142210,0.026780,0.835803,0.838098,1.116743,1.117112,-0.403371,0.963168,-0.2411,-0.0892,-0.092185,-0.213594,0.851034,0.851033,-0.069289,-0.054665,-0.116693,-0.101908,-0.114477,-0.040545,-0.382269,-0.198759,-0.130167,-0.111626,-0.665297,-0.183971,-0.666516,-0.353639,-0.429809,-0.420068,-0.519454,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.901521,2.484628,-0.042885,-0.070096,-0.224592,-0.425967,-0.041412,-0.296357,-0.335980,-0.388766,-0.319231,-1.196466,-1.193186,-0.887684,-0.887548,0.050858,-0.597711,-0.2411,-0.0892,-0.093555,-0.220491,-1.169038,-1.169039,-0.069263,-0.054789,-0.132208,-0.124329,-0.119933,-0.040545,-0.382269,-0.198759,-0.130167,-0.111626,0.534508,0.000853,-0.176512,0.499331,-0.429809,-0.420068,-0.164155,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.714985,-0.300657,-0.042692,-0.011842,-0.211224,-0.425967,-0.041412,-0.262165,0.344916,-0.202924,-0.203894,0.835803,0.838098,-0.370227,1.143510,0.524837,-0.418984,-0.2411,-0.0892,-0.092834,-0.220474,0.874123,0.874123,-0.069336,-0.054753,-0.118651,-0.105175,-0.114684,-0.040545,-0.382269,-0.198759,-0.130167,-0.111626,-0.019248,-0.276383,-0.299013,-0.353639,-0.429809,-0.420068,-0.341805,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


### 7. Creating pipelines for models

In [19]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])

### 8. Model training

In [ ]:
lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

lr_pred = lr_pipeline.predict(X_test)
rf_pred = rf_pipeline.predict(X_test)

### 9. Model Evaluation

In [21]:
print("Logistic Regression")
print("Accuracy :", accuracy_score(y_test, lr_pred))
print("Precision:", precision_score(y_test, lr_pred))
print("Recall   :", recall_score(y_test, lr_pred))
print("F1 Score :", f1_score(y_test, lr_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, lr_pred))

print("\nRandom Forest")
print("Accuracy :", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall   :", recall_score(y_test, rf_pred))
print("F1 Score :", f1_score(y_test, rf_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

Logistic Regression
Accuracy : 0.9896084722741678
Precision: 0.9481132075471698
Recall   : 0.9708783246856448
F1 Score : 0.9593607341088076
Confusion Matrix:
[[440421   3410]
 [  1869  62310]]

Random Forest
Accuracy : 0.9962736953996969
Precision: 0.9875540108961112
Recall   : 0.9828916000560931
F1 Score : 0.9852172894459412
Confusion Matrix:
[[443036    795]
 [  1098  63081]]


### 10. Overfitting Check

In [38]:
rf_train_pred = rf_pipeline.predict(X_train)
rf_test_pred = rf_pipeline.predict(X_test)

print("Train Accuracy:", accuracy_score(y_train, rf_train_pred))
print("Test Accuracy :", accuracy_score(y_test, rf_test_pred))

print("Train F1:", f1_score(y_train, rf_train_pred))
print("Test F1 :", f1_score(y_test, rf_test_pred))

Train Accuracy: 0.9999990157659531
Test Accuracy : 0.9962736953996969
Train F1: 0.9999961105082399
Test F1 : 0.9852172894459412


## MODEL B ( High Correlation Features )